# Inspect estimation pickles

Walks through every `start_NN.pkl` (a `pyblp.ProblemResults`) in `output/seed_0/estimates/`, plus the matching `truth.pkl`.

**Kernel:** select the project's uv venv — `./.venv/bin/python` (or `Python 3.x ('.venv')` in the kernel picker). `pyblp` must be importable to unpickle these objects.

In [10]:
import os, pickle, glob
import numpy as np
import pandas as pd
import pyblp  # required for unpickling ProblemResults

SEED_DIR = "output/seed_0"
ESTIMATES_DIR = os.path.join(SEED_DIR, "estimates")

with open(os.path.join(SEED_DIR, "truth.pkl"), "rb") as fh:
    truth = pickle.load(fh)

pkls = sorted(glob.glob(os.path.join(ESTIMATES_DIR, "start_*.pkl")))
results = {}
for p in pkls:
    sid = int(os.path.basename(p).split("_")[1].split(".")[0])
    with open(p, "rb") as fh:
        results[sid] = pickle.load(fh)

print(f"loaded {len(results)} ProblemResults from {ESTIMATES_DIR}")
print(f"truth keys: {list(truth.keys())}")
print(f"truth.seed = {truth.get('seed')!r}, T={truth['T']}, J={truth['J']}")

loaded 10 ProblemResults from output/seed_0/estimates
truth keys: ['beta', 'sigma', 'pi', 'gamma', 'xi', 'omega', 'T', 'J', 'seed']
truth.seed = 0, T=200, J=10


## 1. Cross-start summary

GMM objective, optimizer status, and wall time for each start. Lowest-objective converged start = the one used for the recovery plot.

In [11]:
rows = []
for sid, r in results.items():
    rows.append({
        "start_id": sid,
        "step": int(r.step),
        "objective": float(r.objective),
        "converged": bool(r.converged),
        "obj_evals": int(r.cumulative_objective_evaluations),
        "opt_iters": int(r.cumulative_optimization_iterations),
        "total_sec": float(r.cumulative_total_time),
        "grad_max": float(np.abs(r.gradient).max()),
        "markets_fp_ok": int(r.fp_converged[:, -1].sum()),
    })
summary = pd.DataFrame(rows).sort_values("objective").reset_index(drop=True)
summary

,start_id,step,objective,converged,obj_evals,opt_iters,total_sec,grad_max,markets_fp_ok
0,7,2,32.293243,True,80,67,101.331282,1.997136e-06,200
1,8,2,32.293247,True,75,61,256.160145,2.003191e-06,200
2,9,2,32.590018,True,67,54,88.309826,2.669544e-06,200
3,3,2,32.590035,True,61,50,84.231118,2.686708e-06,200
4,6,2,32.590053,True,65,51,192.816435,2.682020e-06,200
5,0,2,32.998399,True,59,46,82.304126,1.747651e-06,200
6,2,2,32.998407,True,62,45,85.196720,9.668501e-06,200
7,4,2,32.998415,True,61,46,87.825596,2.751270e-06,200
8,5,2,32.998429,True,59,45,83.114126,9.845245e-07,200
9,1,2,70.586268,True,88,74,104.428537,3.603320e-06,200


In [12]:
best_id = int(summary.iloc[0]["start_id"])
best = results[best_id]
print(f"best converged start: #{best_id}  objective={best.objective:.6g}")
best  # rich repr from pyblp

best converged start: #7  objective=32.2932


Problem Results Summary:
GMM     Objective      Gradient         Hessian         Hessian     Clipped  Weighting Matrix  Covariance Matrix
Step      Value          Norm       Min Eigenvalue  Max Eigenvalue  Shares   Condition Number  Condition Number 
----  -------------  -------------  --------------  --------------  -------  ----------------  -----------------
 2    +3.229324E+01  +1.997136E-06  +5.031332E+00   +1.035804E+03      0      +1.895718E+05      +1.035371E+07  

Cumulative Statistics:
Computation  Optimizer  Optimization   Objective   Fixed Point  Contraction
   Time      Converged   Iterations   Evaluations  Iterations   Evaluations
-----------  ---------  ------------  -----------  -----------  -----------
 00:01:41       Yes          67           80          75434       235219   

Nonlinear Coefficient Estimates (Robust SEs in Parentheses):
Sigma:         1             prices             x1               x2               x3         |   Pi:        income             age   

## 2. Truth vs estimate (best start), with 95% CIs

One table per parameter block. `nan` in the `se` column means the entry is structurally fixed (e.g. a zero in `Pi`) and was not estimated.

In [13]:
def block(name, truth_arr, est_arr, se_arr, row_labels=None, col_labels=None):
    truth_arr = np.asarray(truth_arr).flatten()
    est_arr = np.asarray(est_arr).flatten()
    se_arr = np.asarray(se_arr).flatten()
    if row_labels is None:
        labels = [f"{name}_{k}" for k in range(truth_arr.size)]
    else:
        labels = row_labels
    df = pd.DataFrame({
        "param": labels,
        "truth": truth_arr,
        "estimate": est_arr,
        "se": se_arr,
        "abs_err": np.abs(est_arr - truth_arr),
        "z": (est_arr - truth_arr) / se_arr,
    })
    return df

beta_labels = ["const", "prices (alpha)", "x1", "x2", "x3", "x4", "x5"]
gamma_labels = ["const", "x1", "x2", "w1", "w2"]
block("beta", truth["beta"], best.beta, best.beta_se, row_labels=beta_labels)

,param,truth,estimate,se,abs_err,z
0,const,-1.0,-0.861302,0.641590,0.138698,0.216178
1,prices (alpha),-1.5,-1.503403,0.595868,0.003403,-0.005711
2,x1,1.0,1.023733,0.096196,0.023733,0.246718
3,x2,0.8,0.807434,0.075835,0.007434,0.098030
4,x3,0.6,-0.066162,0.524684,0.666162,-1.269644
5,x4,-0.4,-0.418577,0.016763,0.018577,-1.108219
6,x5,0.3,0.293327,0.016813,0.006673,-0.396899


In [15]:
# Sigma diagonal (off-diagonal is identically zero by construction)
sigma_labels = ["const", "prices", "x1", "x2", "x3"]
block("sigma", np.diag(truth["sigma"]), np.diag(best.sigma),
      np.diag(best.sigma_se), row_labels=sigma_labels)

,param,truth,estimate,se,abs_err,z
0,const,0.5,0.332306,0.834335,0.167694,-0.200991
1,prices,0.3,0.471254,0.145610,0.171254,1.176115
2,x1,0.4,0.059478,1.795216,0.340522,-0.189683
3,x2,0.3,0.105861,2.076482,0.194139,-0.093494
4,x3,0.2,-0.031218,3.028451,0.231218,-0.076349


In [16]:
# Pi: rows = nonlinear chars, cols = demographics
nl = ["const", "prices", "x1", "x2", "x3"]
demo = ["income", "age", "hh_size", "educ"]
pi_labels = [f"{r} x {c}" for r in nl for c in demo]
pi_df = block("pi", truth["pi"], best.pi, best.pi_se, row_labels=pi_labels)
pi_df  # 20 rows; NaN se = pinned at zero

,param,truth,estimate,se,abs_err,z
0,const x income,0.0,0.000000,NaN,0.000000,NaN
1,const x age,0.2,0.290524,0.575467,0.090524,0.157306
2,const x hh_size,0.0,0.000000,NaN,0.000000,NaN
3,const x educ,0.0,0.000000,NaN,0.000000,NaN
4,prices x income,-0.2,-0.291813,0.458500,0.091813,-0.200247
5,prices x age,0.0,0.000000,NaN,0.000000,NaN
6,prices x hh_size,0.0,0.000000,NaN,0.000000,NaN
7,prices x educ,0.0,0.000000,NaN,0.000000,NaN
8,x1 x income,0.0,0.000000,NaN,0.000000,NaN
9,x1 x age,0.0,0.000000,NaN,0.000000,NaN


In [17]:
block("gamma", truth["gamma"], best.gamma, best.gamma_se, row_labels=gamma_labels)

,param,truth,estimate,se,abs_err,z
0,const,1.0,1.005961,0.108880,0.005961,0.054751
1,x1,0.3,0.277159,0.013766,0.022841,-1.659270
2,x2,0.2,0.181450,0.010038,0.018550,-1.847922
3,w1,0.4,0.381218,0.017829,0.018782,-1.053488
4,w2,0.3,0.285145,0.014380,0.014855,-1.033013


## 3. Per-start stability

A wide DataFrame: each row a parameter, each column a start. Lets you eyeball which parameters are stable across starts and which jump basins.

In [18]:
def flat(r):
    out = {}
    sigma, pi, beta, gamma = r.sigma, r.pi, r.beta, r.gamma
    for k in range(sigma.shape[0]):
        out[f"sigma_{k}_{k}"] = float(sigma[k, k])
    for k in range(pi.shape[0]):
        for d in range(pi.shape[1]):
            out[f"pi_{k}_{d}"] = float(pi[k, d])
    for k, v in enumerate(np.asarray(beta).flatten()):
        out[f"beta_{k}"] = float(v)
    for k, v in enumerate(np.asarray(gamma).flatten()):
        out[f"gamma_{k}"] = float(v)
    return out

truth_flat = {**{f"sigma_{k}_{k}": float(truth["sigma"][k, k]) for k in range(5)},
              **{f"pi_{k}_{d}": float(truth["pi"][k, d]) for k in range(5) for d in range(4)},
              **{f"beta_{k}": float(v) for k, v in enumerate(truth["beta"])},
              **{f"gamma_{k}": float(v) for k, v in enumerate(truth["gamma"])}}

wide = pd.DataFrame({sid: flat(r) for sid, r in sorted(results.items())})
wide.insert(0, "truth", pd.Series(truth_flat))
wide  # 37 parameters x (1 truth + n_starts) columns

,truth,0,1,2,3,4,5,6,7,8,9
sigma_0_0,0.5,0.321450,-0.062518,0.321449,0.045913,0.321448,0.321447,0.045912,0.332306,0.332306,0.045911
sigma_1_1,0.3,0.471688,0.023959,0.471688,0.507788,0.471689,0.471689,0.507788,0.471254,0.471255,0.507788
sigma_2_2,0.4,0.056558,0.084568,0.056558,0.053586,0.056558,0.056558,0.053586,0.059478,0.059477,0.053586
sigma_3_3,0.3,0.100948,-0.001711,0.100947,0.112049,0.100947,0.100947,0.112049,0.105861,0.105860,0.112049
sigma_4_4,0.2,-0.031976,-0.307556,-0.031976,0.015810,-0.031976,-0.031976,0.015810,-0.031218,-0.031218,0.015810
pi_0_0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
pi_0_1,0.2,0.293973,0.172336,0.293973,0.079382,0.293971,0.293970,0.079381,0.290524,0.290523,0.079381
pi_0_2,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
pi_0_3,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
pi_1_0,-0.2,-0.289935,-0.327089,-0.289935,0.158970,-0.289935,-0.289935,0.158970,-0.291813,-0.291813,0.158970


In [ ]:
# Spread across starts: max-min of each parameter, for the converged ones.
converged_ids = summary.loc[summary["converged"], "start_id"].tolist()
spread = wide[converged_ids].agg(["min", "max", "std"]).T
spread["range"] = spread["max"] - spread["min"]
spread.sort_values("range", ascending=False).head(15)

## 4. Drill into a single ProblemResults

Change `inspect_id` to look at any other start.

In [19]:
inspect_id = best_id  # set to any start_id present in `results`
r = results[inspect_id]

print(f"step              : {r.step}")
print(f"objective         : {float(r.objective):.6g}")
print(f"converged         : {bool(r.converged)}")
print(f"obj evaluations   : {r.cumulative_objective_evaluations}")
print(f"opt iterations    : {r.cumulative_optimization_iterations}")
print(f"total time (s)    : {r.cumulative_total_time:.2f}")
print(f"gradient max|.|   : {float(np.abs(r.gradient).max()):.3e}")
T_total = r.fp_converged.shape[0]
markets_final_ok = int(r.fp_converged[:, -1].sum())
print(f"markets fp-conv   : {markets_final_ok} / {T_total} (at final evaluation)")
print(f"mean contraction  : {r.cumulative_fp_iterations.mean():.1f} iters/market/eval")

step              : 2
objective         : 32.2932
converged         : True
obj evaluations   : 80
opt iterations    : 67
total time (s)    : 101.33
gradient max|.|   : 1.997e-06
markets fp-conv   : 200 / 200 (at final evaluation)
mean contraction  : 4.7 iters/market/eval


In [20]:
# All non-private attributes / methods available on the object.
[a for a in dir(r) if not a.startswith("_")]

['W',
 'beta',
 'beta_bounds',
 'beta_labels',
 'beta_se',
 'bootstrap',
 'clipped_costs',
 'clipped_shares',
 'compute_agent_scores',
 'compute_aggregate_elasticities',
 'compute_approximate_prices',
 'compute_consumer_surpluses',
 'compute_costs',
 'compute_delta',
 'compute_demand_hessians',
 'compute_demand_jacobians',
 'compute_diversion_ratios',
 'compute_elasticities',
 'compute_hhi',
 'compute_long_run_diversion_ratios',
 'compute_markups',
 'compute_micro_scores',
 'compute_micro_values',
 'compute_optimal_instruments',
 'compute_passthrough',
 'compute_prices',
 'compute_probabilities',
 'compute_profit_hessians',
 'compute_profits',
 'compute_shares',
 'contraction_evaluations',
 'converged',
 'cumulative_contraction_evaluations',
 'cumulative_converged',
 'cumulative_fp_converged',
 'cumulative_fp_iterations',
 'cumulative_objective_evaluations',
 'cumulative_optimization_iterations',
 'cumulative_optimization_time',
 'cumulative_total_time',
 'delta',
 'extract_diagonal_me

In [21]:
# Examples of derived quantities you can compute from a ProblemResults.
# Comment / uncomment as needed.

# r.compute_diversion_ratios()    # post-price-change substitution
# r.compute_markups()             # p - mc
# r.compute_costs()               # marginal costs implied by FOCs
# r.compute_consumer_surpluses()  # per-market CS

# Quick example: mean own-price elasticity across products.
# compute_elasticities() returns (T*J, J): rows stack markets t=0..T-1, each
# row k gives d log s_k / d log p_j across the J products in that market.
# Own elasticity = diagonal of each per-market (J, J) block.
el = r.compute_elasticities()
T, J = truth["T"], truth["J"]
el_blocks = el.reshape(T, J, J)
own = np.diagonal(el_blocks, axis1=1, axis2=2).flatten()  # shape (T*J,)
print(f"mean own-price elasticity   : {own.mean():.3f}")
print(f"median own-price elasticity : {np.median(own):.3f}")
print(f"share inelastic (|own| < 1) : {(np.abs(own) < 1).mean()*100:.1f}%")

Computing elasticities with respect to prices ...
Finished after 00:00:00.

mean own-price elasticity   : -3.324
median own-price elasticity : -3.338
share inelastic (|own| < 1) : 0.0%
